# 수면 기반 피로도 예측
시작: 2026-10-09 14:45 KST · 종료: 16:35 KST
문제·채점 기준은 README.md를 확인하세요. 아래 기본 예제를 개선하고 submission.csv를 저장하세요.
서버 전체에서 커널 하나만 사용합니다. 완료 후 Kernel → Shutdown으로 실행 자리를 반납하세요.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
DATA=Path('data')
train=pd.read_csv(DATA/'train.csv'); test=pd.read_csv(DATA/'test.csv'); sample=pd.read_csv(DATA/'sample_submission.csv')
print(train.shape, test.shape)


(80, 5) (21, 4)


## 풀이: 하한 30 에서 검열된 선형 회귀 (Tobit)
- **데이터 구조**: 학습 80행 중 15행(19%)의 `fatigue_score`가 정확히 30.0 입니다. 30 초과 행만으로 선형 적합을 하면 이 15행의 예측값이 대부분 30 미만(16~33)으로 나와, 타깃이 `max(30, 선형 + 잡음)` 형태로 만들어졌다고 판단했습니다.
- **모델**: 30 이하를 "검열된 관측"으로 다루는 최대우도 선형 회귀(Tobit)를 직접 구현했습니다(scipy). 예측은 `E[max(30, mu + eps)]` 입니다.
- **검증**(5-Fold x 20회 반복 RMSE): 제공 베이스라인 RF 5.12, 선형 3.74, 선형+하한 클리핑 3.26, **Tobit 2.84**. Tobit 은 20회 반복 모두에서 선형+클리핑보다 낮았습니다.
- 기각: 카페인·야간근무 포화 변환(3.5~4.5), 2차 다항 Ridge(3.39), HGB(5.87).

In [2]:
from sklearn.model_selection import KFold
R = 20   # 5-Fold 를 20번 반복 (학습 80행뿐이라 단일 분할 점수는 변동이 큼)
def cv_rmse(X, y, make, post=None):
    out = []
    for r in range(R):
        oof = np.zeros(len(y))
        for a, b in KFold(5, shuffle=True, random_state=r).split(X):
            p = make().fit(X.iloc[a], y.iloc[a]).predict(X.iloc[b]); oof[b] = post(p) if post else p
        out.append(np.sqrt(np.mean((y.values - oof) ** 2)))
    return np.array(out)
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.linear_model import LinearRegression

class Tobit:
    """하한 L 에서 검열된 선형 회귀(최대우도). 예측 = E[max(L, mu + eps)]"""
    def __init__(self, L=30.0): self.L = L
    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y, float); self.m, self.s = X.mean(0), X.std(0) + 1e-9
        Z = np.c_[np.ones(len(X)), (X - self.m) / self.s]; cen = y <= self.L + 1e-9
        b0 = np.linalg.lstsq(Z, y, rcond=None)[0]; s0 = np.log(max(np.std(y - Z @ b0), 1e-2))
        def nll(w):
            b, sg = w[:-1], np.exp(w[-1]); mu = Z @ b
            return -(norm.logpdf((y[~cen] - mu[~cen]) / sg).sum() - (~cen).sum() * np.log(sg) + norm.logcdf((self.L - mu[cen]) / sg).sum())
        r = minimize(nll, np.r_[b0, s0], method="BFGS"); self.b, self.sg = r.x[:-1], float(np.exp(r.x[-1])); return self
    def predict(self, X):
        Z = np.c_[np.ones(len(X)), (np.asarray(X, float) - self.m) / self.s]; mu = Z @ self.b; z = (mu - self.L) / self.sg
        return self.L + (mu - self.L) * norm.cdf(z) + self.sg * norm.pdf(z)

features = [c for c in test.columns if c != "soldier_id"]; X, y = train[features], train["fatigue_score"]
print("fatigue_score == 30.0 인 행:", int((y == 30).sum()), "/", len(y))
a = cv_rmse(X, y, LinearRegression); b = cv_rmse(X, y, LinearRegression, lambda p: np.clip(p, 30, None)); c = cv_rmse(X, y, Tobit)
for n, v in (("선형", a), ("선형 + 하한 30 클리핑", b), ("Tobit (메인)", c)):
    print(f"{n:22s} RMSE {v.mean():.4f} (반복 sd {v.std():.4f}) -> 점수 {100 / (1 + v.mean() / 9.65650938):.2f}")
print("Tobit 이 선형+클리핑보다 낮은 반복:", int((c < b).sum()), "/", R)

model = Tobit().fit(X, y)
print("sigma %.2f" % model.sg)
submission = pd.DataFrame({"soldier_id": test.soldier_id, "fatigue_score": model.predict(test[features])})[sample.columns]
assert np.isfinite(submission.fatigue_score).all()
submission.to_csv("submission.csv", index=False)
print("예측 범위 %.1f ~ %.1f (train %.1f ~ %.1f)" % (submission.fatigue_score.min(), submission.fatigue_score.max(), y.min(), y.max())); submission.head()

fatigue_score == 30.0 인 행: 15 / 80


선형                     RMSE 3.7358 (반복 sd 0.1031) -> 점수 72.10
선형 + 하한 30 클리핑         RMSE 3.2622 (반복 sd 0.0745) -> 점수 74.75
Tobit (메인)             RMSE 2.8430 (반복 sd 0.0581) -> 점수 77.26
Tobit 이 선형+클리핑보다 낮은 반복: 20 / 20
sigma 3.08
예측 범위 30.0 ~ 61.1 (train 30.0 ~ 69.2)


,soldier_id,fatigue_score
0,S001,50.729293
1,S005,32.448461
2,S011,39.958220
3,S013,44.988476
4,S019,55.351461


In [3]:
check = pd.read_csv('submission.csv')
assert check.columns.tolist() == sample.columns.tolist()
assert check.iloc[:,0].tolist() == sample.iloc[:,0].tolist()
assert check.iloc[:,0].is_unique and not check.isna().any().any()
print('제출 형식 확인:', check.shape)


제출 형식 확인: (21, 2)
